# 25. 경제 시계열 프로젝트 시작: API로 데이터 가져오기

이번 장에서는 경제 데이터를 API로 가져오는 첫 흐름을 배웁니다.

목표:

```text
.env에서 API 키 읽기
-> FRED API 요청
-> JSON을 DataFrame으로 변환
-> 날짜와 숫자 정리
-> 그래프 확인
-> CSV 저장
```

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path
import os

# requests는 API 주소로 HTTP 요청을 보내는 라이브러리입니다.
import requests

# pandas는 표 형태 데이터 정리에 사용합니다.
import pandas as pd

# matplotlib.pyplot은 그래프를 그릴 때 사용합니다.
import matplotlib.pyplot as plt

## 2. .env 파일 읽기 함수 만들기

API 키는 코드에 직접 적지 않습니다.

`.env` 파일에서 읽어 환경변수처럼 사용합니다.

In [ ]:
def load_env_file(env_path):
    """간단한 .env 파일을 읽어 os.environ에 넣습니다."""
    
    env_path = Path(env_path)
    
    if not env_path.exists():
        print("env 파일이 없습니다:", env_path)
        return
    
    for line in env_path.read_text(encoding="utf-8").splitlines():
        # 빈 줄이나 주석은 건너뜁니다.
        if not line.strip() or line.strip().startswith("#"):
            continue
        
        # KEY=VALUE 형태만 처리합니다.
        if "=" not in line:
            continue
        
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        
        # 이미 환경변수에 값이 있으면 덮어쓰지 않습니다.
        os.environ.setdefault(key, value)

## 3. API 키 확인

In [ ]:
# C:\work\.env에 키가 있다고 가정합니다.
ENV_PATH = Path(r"C:\work\.env")

load_env_file(ENV_PATH)

# getenv()는 환경변수 값을 읽습니다.
# 여기서는 키 값 자체를 출력하지 않고, 존재 여부만 확인합니다.
fred_api_key = os.getenv("FRED_API_KEY")

print("FRED_API_KEY 존재 여부:", fred_api_key is not None)

## 4. FRED API 요청 함수 만들기

FRED의 `series/observations` 엔드포인트를 사용하면 특정 경제 지표의 관측값을 가져올 수 있습니다.

예시 series id:

```text
CPIAUCSL = 소비자물가지수
FEDFUNDS = 연방기금금리
UNRATE   = 실업률
```

In [ ]:
def fetch_fred_series(series_id, api_key, observation_start="2000-01-01"):
    """FRED에서 특정 series_id의 시계열 관측값을 가져와 DataFrame으로 반환합니다."""
    
    url = "https://api.stlouisfed.org/fred/series/observations"
    
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": observation_start,
    }
    
    # requests.get()은 URL에 GET 요청을 보냅니다.
    response = requests.get(url, params=params, timeout=30)
    
    # raise_for_status()는 HTTP 오류가 있으면 예외를 발생시킵니다.
    response.raise_for_status()
    
    # json()은 응답 본문을 파이썬 딕셔너리로 변환합니다.
    data = response.json()
    
    # observations 안에 실제 관측값 목록이 들어 있습니다.
    observations = data["observations"]
    
    df = pd.DataFrame(observations)
    
    return df

## 5. CPI 데이터 가져오기

In [ ]:
series_id = "CPIAUCSL"

# API 키가 없으면 이 셀은 실행되지 않도록 확인합니다.
if fred_api_key is None:
    raise ValueError("FRED_API_KEY가 없습니다. C:\\work\\.env를 확인하세요.")

cpi_raw = fetch_fred_series(
    series_id=series_id,
    api_key=fred_api_key,
    observation_start="2000-01-01"
)

print("raw shape:", cpi_raw.shape)
cpi_raw.head()

## 6. 날짜와 값 정리하기

API 응답에서는 값이 문자열로 들어올 수 있습니다.

딥러닝이나 그래프에 사용하려면 날짜는 datetime, 값은 숫자로 바꿔야 합니다.

In [ ]:
def clean_fred_dataframe(raw_df, value_name):
    """FRED 원본 DataFrame에서 date와 value를 정리합니다."""
    
    cleaned = raw_df[["date", "value"]].copy()
    
    # 날짜 문자열을 datetime으로 변환합니다.
    cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
    
    # value 컬럼은 문자열일 수 있으므로 숫자로 변환합니다.
    # errors="coerce"는 변환 불가능한 값을 NaN으로 바꿉니다.
    cleaned[value_name] = pd.to_numeric(cleaned["value"], errors="coerce")
    
    # 원래 value 문자열 컬럼은 제거합니다.
    cleaned = cleaned.drop(columns=["value"])
    
    # 결측치를 제거합니다.
    cleaned = cleaned.dropna(subset=["date", value_name])
    
    # 날짜 순서대로 정렬합니다.
    cleaned = cleaned.sort_values("date")
    
    return cleaned


cpi_df = clean_fred_dataframe(cpi_raw, value_name="cpi")

print("cleaned shape:", cpi_df.shape)
cpi_df.head()

## 7. CPI 그래프 그리기

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(cpi_df["date"], cpi_df["cpi"])
plt.title("FRED CPIAUCSL")
plt.xlabel("Date")
plt.ylabel("CPI")
plt.tight_layout()
plt.show()

## 8. 여러 경제 지표 가져오기

같은 함수로 여러 지표를 가져올 수 있습니다.

이번 예시는 CPI, 실업률, 연방기금금리입니다.

In [ ]:
series_map = {
    "CPIAUCSL": "cpi",
    "UNRATE": "unemployment_rate",
    "FEDFUNDS": "fed_funds_rate",
}

series_frames = []

for series_id, column_name in series_map.items():
    raw = fetch_fred_series(
        series_id=series_id,
        api_key=fred_api_key,
        observation_start="2000-01-01"
    )
    
    cleaned = clean_fred_dataframe(raw, value_name=column_name)
    series_frames.append(cleaned)
    
    print(series_id, cleaned.shape)

## 9. 날짜 기준으로 합치기

여러 경제 지표를 함께 쓰려면 날짜를 기준으로 합쳐야 합니다.

In [ ]:
# 첫 번째 데이터프레임부터 시작합니다.
economy_df = series_frames[0]

# 나머지 데이터프레임을 date 기준으로 차례대로 병합합니다.
for frame in series_frames[1:]:
    economy_df = economy_df.merge(frame, on="date", how="inner")

print("merged shape:", economy_df.shape)
economy_df.head()

## 10. 여러 지표 그래프 보기

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(economy_df["date"], economy_df["cpi"])
axes[0].set_title("CPI")

axes[1].plot(economy_df["date"], economy_df["unemployment_rate"])
axes[1].set_title("Unemployment Rate")

axes[2].plot(economy_df["date"], economy_df["fed_funds_rate"])
axes[2].set_title("Federal Funds Rate")

plt.tight_layout()
plt.show()

## 11. CSV로 저장하기

In [ ]:
OUTPUT_DIR = Path(r"C:\work\deepLearning\deepLearning_textbook\data_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "fred_economic_indicators.csv"

# to_csv()는 DataFrame을 CSV 파일로 저장합니다.
economy_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)

## 12. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. API는 코드로 외부 서비스에서 데이터를 가져오는 통로다.
2. API 키는 .env로 관리하고 코드에 직접 쓰지 않는다.
3. FRED series/observations API로 경제 시계열을 가져올 수 있다.
4. JSON 응답은 pandas DataFrame으로 변환해 정리한다.
5. 날짜는 datetime, 값은 숫자로 변환해야 한다.
6. 수집한 데이터는 재현성을 위해 CSV로 저장해 둘 수 있다.
```

## 과제

1. `series_map`에 다른 FRED 지표를 하나 추가해보세요.
2. API 키를 출력하지 않아야 하는 이유를 적어보세요.
3. 경제 시계열에서 날짜 기준 병합이 왜 중요한지 설명해보세요.
4. 다음 장에서 어떤 경제 지표를 예측 대상으로 삼고 싶은지 적어보세요.